### Identify Cell Wall Genes in Non-Arabidopsis Genomes With HMMR3
Grady King

The .xlsx file comes from a literature review in [this paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC10762515/#kiad267-B46) that found all genes in Arabidopsis that relate to cell wall biosynthesis. 

my goal is to build a model that takes these genes, builds a "profile" of how these proteins look, then I can use the model to predict which genes in a different genome fit with the profile, which are likely also related to cell walls. this file is the laziest possible way of doing that, using the off-the-shelf pyhmmer that calls HMMER3, then I can build from there.

In [7]:
import pyhmmer
import pandas as pd
import numpy as np

In [49]:
table_s8 = pd.read_excel("kiad267_supplementary_data.xlsx", sheet_name="Supplemental Table S8", header=1)

In [9]:
print(table_s8)

     Arabidopsis thaliana gene ID (TAIR assembly) Cell wall gene function  \
0                                       At1g05570                    CALS   
1                                       At2g36850                    CALS   
2                                       At4g04970                    CALS   
3                                       At4g03550                    CALS   
4                                       At2g31960                    CALS   
...                                           ...                     ...   
1308                                    At1g62700                    VND5   
1309                                    At5g62380                    VND6   
1310                                    At1g71930                    VND7   
1311                                    At2g44745                  WRKY12   
1312                                    At5g64530                    XND1   

      Arabidopsis thaliana chromosome (TAIR assembly) Start (bp, BC_GFF)  \

The Supplemental Table S8 contains each arabidopsis gene related to cell wall biosynthesis, along with basic information about it, which pfam protein families it belongs to, and its sequence. 

Let's take the pfam values at face value and download all of those profile HMMs. Then, we could say that if a given protein from a new genome matches one of the profiles from arabidopsis, then it must be a cell wall gene. Exactly where the cutoff will be can be decided later.

In [10]:
# isolate pfam IDs so I can download them all

split_pfam = set(table_s8["PFAM (Hmmsearch, E1)"].str.split(';').explode())
split_pfam.remove('')
split_pfam.remove(np.nan)
split_pfam

{'PF00004',
 'PF00005',
 'PF00009',
 'PF00025',
 'PF00046',
 'PF00067',
 'PF00069',
 'PF00071',
 'PF00106',
 'PF00107',
 'PF00141',
 'PF00150',
 'PF00168',
 'PF00169',
 'PF00182',
 'PF00201',
 'PF00221',
 'PF00225',
 'PF00232',
 'PF00235',
 'PF00249',
 'PF00294',
 'PF00295',
 'PF00326',
 'PF00331',
 'PF00332',
 'PF00337',
 'PF00342',
 'PF00349',
 'PF00350',
 'PF00394',
 'PF00408',
 'PF00501',
 'PF00514',
 'PF00534',
 'PF00535',
 'PF00544',
 'PF00553',
 'PF00560',
 'PF00561',
 'PF00642',
 'PF00652',
 'PF00722',
 'PF00735',
 'PF00759',
 'PF00777',
 'PF00847',
 'PF00862',
 'PF00891',
 'PF00892',
 'PF00933',
 'PF00984',
 'PF01031',
 'PF01055',
 'PF01061',
 'PF01073',
 'PF01095',
 'PF01135',
 'PF01190',
 'PF01262',
 'PF01301',
 'PF01357',
 'PF01370',
 'PF01501',
 'PF01565',
 'PF01596',
 'PF01636',
 'PF01697',
 'PF01704',
 'PF01738',
 'PF01762',
 'PF01915',
 'PF01926',
 'PF02018',
 'PF02065',
 'PF02129',
 'PF02140',
 'PF02183',
 'PF02212',
 'PF02263',
 'PF02318',
 'PF02319',
 'PF02364',
 'PF

In [11]:
# download all hmm files using a for loop, requesting from interpro (the database behind pfam)
import requests
import os
import time
from tqdm import tqdm

os.makedirs("hmms", exist_ok = True)

for pfam_id in tqdm(list(split_pfam)):
    if not os.path.exists(f"hmms/{pfam_id}.hmm.gz"):
        with open(f"hmms/{pfam_id}.hmm.gz", "wb+") as f:
            with requests.get(f"https://www.ebi.ac.uk/interpro/api/entry/pfam/{pfam_id}?annotation=hmm") as r:
                f.write(r.content)
        time.sleep(0.25)

100%|██████████| 292/292 [00:00<00:00, 496850.62it/s]


In [52]:
hmms = []
valid_hmm_paths = []
paths = [x for x in os.listdir('hmms') if x.endswith(".hmm.gz")]
for path in paths:
    try:
        with pyhmmer.plan7.HMMFile(f"hmms/{path}", alphabet=pyhmmer.easel.Alphabet.amino()) as hmm:
            hmms.extend(hmm)
            valid_hmm_paths.append(path)
    except:
        continue

print(len(hmms), "of", len(paths))


gzip: hmms/PF16863.hmm.gz: not in gzip format

gzip: hmms/PF05664.hmm.gz: not in gzip format

gzip: hmms/PF07526.hmm.gz: unexpected end of file

gzip: hmms/PF04048.hmm.gz: not in gzip format

gzip: hmms/PF03360.hmm.gz: unexpected end of file


286 of 292



gzip: hmms/PF02439.hmm.gz: not in gzip format


In [53]:
print(hmms[0].name, hmms[0].alphabet)

AMP-binding AA()


In [44]:
pyhmmer.hmmpress(hmms, 'hmms/pressed')

286

make a table converting the name of the hmm to the genes associated with it

In [50]:
# convert table_s8 to have each PFAM on its own row
table_s8["PFAM (Hmmsearch, E1)"] = table_s8["PFAM (Hmmsearch, E1)"].str.split(';')
table_s8 = table_s8.explode("PFAM (Hmmsearch, E1)")
table_s8.head()

,Arabidopsis thaliana gene ID (TAIR assembly),Cell wall gene function,Arabidopsis thaliana chromosome (TAIR assembly),"Start (bp, BC_GFF)","End (bp, BC_GFF)",Cell wall gene broad process,"PFAM (Hmmsearch, E1)",Sequence (BC_PEP)
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF02364,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF04652,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF14288,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...
1,At2g36850,CALS,2,15454671,15469964,Callose_synthesis,PF02364,MARVYSNWDRLVRATLRREQLRNTGQGHERVSSGLAGAVPPSLGRA...


In [ ]:
# remove .hmm.gz from valid paths, should be same length as hmm list and same order
pfam_ids = [x[:-7] for x in valid_hmm_paths]
print(len(pfam_ids), len(hmms))

286 286


In [ ]:
pfam_id_to_hmm_id = {}
for ID, hmm in zip(pfam_ids, hmms):
    pfam_id_to_hmm_id[ID] = hmm.name

# display first three items to show what the dict is
dict(list(pfam_id_to_hmm_id.items())[0:3])

{'PF00501': 'AMP-binding', 'PF11841': 'ELMO_ARM', 'PF05048': 'NosD'}

In [84]:
table_s8 = table_s8.filter(cond=pd.col("PFAM (Hmmsearch, E1)") != "")
table_s8 = table_s8.filter(cond=pd.col("PFAM (Hmmsearch, E1)").isin(pfam_id_to_hmm_id))
table_s8

,Arabidopsis thaliana gene ID (TAIR assembly),Cell wall gene function,Arabidopsis thaliana chromosome (TAIR assembly),"Start (bp, BC_GFF)","End (bp, BC_GFF)",Cell wall gene broad process,"PFAM (Hmmsearch, E1)",Sequence (BC_PEP)
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF02364,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF04652,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF14288,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...
1,At2g36850,CALS,2,15454671,15469964,Callose_synthesis,PF02364,MARVYSNWDRLVRATLRREQLRNTGQGHERVSSGLAGAVPPSLGRA...
1,At2g36850,CALS,2,15454671,15469964,Callose_synthesis,PF14288,MARVYSNWDRLVRATLRREQLRNTGQGHERVSSGLAGAVPPSLGRA...
...,...,...,...,...,...,...,...,...
1308,At1g62700,VND5,1,23215953,23219209,Transcription_factor,PF02365,MNSFSQVPPGFRFHPTDEELVDYYLRKKVASKRIEIDIIKDVDLYK...
1309,At5g62380,VND6,5,25050152,25052054,Transcription_factor,PF02365,MESLAHIPPGYRFHPTDEELVDYYLKNKVAFPGMQVDVIKDVDLYK...
1310,At1g71930,VND7,1,27074702,27077991,Transcription_factor,PF02365,MDNIMQSSMPPGFRFHPTEEELVGYYLDRKINSMKSALDVIVEIDL...
1311,At2g44745,WRKY12,2,18447203,18449093,Transcription_factor,PF03106,MEGGGRRVFSNYDLQQVTSSSTTIQENMNFLVPFEETNVLTFFSSS...


In [88]:
table_s8['HMM_id'] = [pfam_id_to_hmm_id[x] for x in table_s8["PFAM (Hmmsearch, E1)"]]
table_s8.head()

,Arabidopsis thaliana gene ID (TAIR assembly),Cell wall gene function,Arabidopsis thaliana chromosome (TAIR assembly),"Start (bp, BC_GFF)","End (bp, BC_GFF)",Cell wall gene broad process,"PFAM (Hmmsearch, E1)",Sequence (BC_PEP),HMM_id
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF02364,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...,Glucan_synthase
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF04652,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...,Vta1
0,At1g05570,CALS,1,1647575,1659611,Callose_synthesis,PF14288,MAQRREPDPPPPQRRILRTQTVGSLGEAMLDSEVVPSSLVEIAPIL...,FKS1_dom1
1,At2g36850,CALS,2,15454671,15469964,Callose_synthesis,PF02364,MARVYSNWDRLVRATLRREQLRNTGQGHERVSSGLAGAVPPSLGRA...,Glucan_synthase
1,At2g36850,CALS,2,15454671,15469964,Callose_synthesis,PF14288,MARVYSNWDRLVRATLRREQLRNTGQGHERVSSGLAGAVPPSLGRA...,FKS1_dom1


In [94]:
# make dict for hmm_id -> arabidopsis gene
hmm_to_arab_gene = {}
for index, row in table_s8.iterrows():
    hmm_id = row['HMM_id']
    arab_id = row['Arabidopsis thaliana gene ID (TAIR assembly)']
    
    if hmm_id not in hmm_to_arab_gene:
        hmm_to_arab_gene[hmm_id] = [arab_id]
    else:
        if arab_id not in hmm_to_arab_gene[hmm_id]:
            hmm_to_arab_gene[hmm_id].append(arab_id)

# display first three items to show what the dict is
dict(list(hmm_to_arab_gene.items())[0:3])

{'Glucan_synthase': ['At1g05570',
  'At2g36850',
  'At4g04970',
  'At4g03550',
  'At2g31960',
  'At5g13000',
  'At5g36870',
  'At2g13680',
  'At3g59100',
  'At1g06490',
  'At3g14570',
  'At3g07160'],
 'Vta1': ['At1g05570',
  'At2g31960',
  'At5g13000',
  'At5g36870',
  'At2g13680',
  'At3g59100',
  'At1g06490',
  'At3g14570'],
 'FKS1_dom1': ['At1g05570',
  'At2g36850',
  'At4g04970',
  'At4g03550',
  'At2g31960',
  'At5g13000',
  'At5g36870',
  'At2g13680',
  'At3g59100',
  'At1g06490',
  'At3g14570',
  'At3g07160']}

In [95]:
import pickle
with open("hmm_to_arab_gene.pkl", "wb+") as f:
    pickle.dump(hmm_to_arab_gene, f)

Now, can compare any genome to the families using `hmmscan`

In [32]:
# example tree genome Fraxinus excelsior obtained from https://www.ncbi.nlm.nih.gov/protein?term=Fraxinus%20excelsior%5BOrganism%5D -> send to: file, FASTA
with pyhmmer.easel.SequenceFile("fraxinus_excelsior.fasta", digital=True) as seq_file:
    sequences = list(seq_file)

len(sequences)

569

In [33]:
with pyhmmer.plan7.HMMPressedFile("hmms/pressed") as hmm_db:
    hits_by_sequence = list(pyhmmer.hmmer.hmmscan(sequences, hmm_db))

print(len(hits_by_sequence))

569


In [24]:
hits_by_sequence[0].incE

0.01

In [6]:
# looks like a list of lists
[list(hits) for hits in hits_by_sequence]

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],


In [7]:
# how many hits are there total?
sum([len(list(hits)) for hits in hits_by_sequence])

66

In [8]:
# what is the histogram of how many hits each sequence gets?
pd.Series([len(list(hits)) for hits in hits_by_sequence]).value_counts()

0    539
1     21
5      5
2      2
8      2
Name: count, dtype: int64

In [28]:
# list out the hits
for hits in hits_by_sequence:
    all_hit = list(hits)
    if len(all_hit) > 0:
        for hit in all_hit:
            # E value is bonferroni-corrected p-value times number of "targets"
            print(hits.query.name, hit.name, hit.score, hit.evalue, hit.pvalue * hits_by_sequence[0].Z, sep = "\t")

XYM53122.1	AAA	39.41424560546875	1.1437173269450989e-11	1.1437173269450989e-11
XYM53102.1	AAA	39.41424560546875	1.1437173269450989e-11	1.1437173269450989e-11
XYM53065.1	AAA	20.949413299560547	5.837232861772399e-06	5.837232861772399e-06
XYM53065.1	AAA_25	17.184280395507812	4.9065420706025395e-05	4.9065420706025395e-05
XYM53065.1	ABC_tran	16.329483032226562	0.000174095821416802	0.000174095821416802
XYM53065.1	AAA_16	15.70114517211914	0.0002551144943242131	0.0002551144943242131
XYM53065.1	RsgA_GTPase	14.241277694702148	0.00044051713144111317	0.00044051713144111317
XYM53043.1	AAA_16	13.03866195678711	0.001678505171293055	0.001678505171293055
XVF99162.1	AAA	39.57913589477539	1.0170620810834244e-11	1.0170620810834244e-11
XVF99142.1	AAA	39.57913589477539	1.0170620810834244e-11	1.0170620810834244e-11
XVF99105.1	AAA	20.949413299560547	5.837232861772399e-06	5.837232861772399e-06
XVF99105.1	AAA_25	17.184280395507812	4.9065420706025395e-05	4.9065420706025395e-05
XVF99105.1	ABC_tran	16.329483032226